# Convolution of Two Square Pulses


> Part of *PHYS 0241: Biomedical Imaging* (Prof. Michael E. Durst, Middlebury College). The [textbook page for this notebook](https://durstlab.middcreate.net/teaching/protected/textbook-biomedicalimaging/python/ch05/ch05-14.html) shows the rendered version together with the pre-rendered animations (course password required).
>
> **Start here:** no setup is needed for this notebook. In Google Colab, choose **File > Save a copy in Drive** to keep your changes.
>
> *Provided for academic use in PHYS 0241; please do not repost or redistribute ([terms of use](https://github.com/durstme/textbook-biomedicalimaging-notebooks/blob/main/LICENSE.md)).*

Two square pulses of width $d$ are slid across each other. At each shift $X$ the
shaded area is the overlap, and collecting those areas traces out the convolution
$g(X)$. The result is a **triangle** of peak $d$ and total width $2d$.

This is the same calculation as `ch05-08`, specialized to the one pair of functions
that gives the cleanest answer. Seeing the seven snapshots stacked makes it obvious
*why* the answer is a triangle: the overlap grows linearly, peaks when the pulses
coincide, and falls linearly away again.

## Import libraries

Only NumPy and Matplotlib. The convolution itself is one call to `np.convolve`, so
nothing beyond the standard scientific stack is needed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
%matplotlib inline
# Define global "bold" style parameters
# This restores the thicker lines and fonts seen in older versions
plot_params = {
    # --- Fonts ---
    'font.size': 14,
    'font.family': 'sans-serif',

    # --- Axes (The Box) ---
    'axes.linewidth': 1.0,

    # --- Lines (The Signal) ---
    'lines.linewidth': 2.5,
}

# Apply these settings globally
plt.rcParams.update(plot_params)


## Define parameters

Three things happen in the cell below.

**The two shapes.** `get_f` and `get_h` both return a rect of width $d$ centered on the
origin. Defining them as *functions* rather than arrays matters: the plotting code needs
to evaluate $h(X-x)$ at many different shifts $X$, which is far easier from a function
than by re-indexing an array.

**The convolution, once.** `np.convolve(..., mode='full')` computes the whole result in a
single call. The multiplication by `dt` is what turns a discrete sum into an approximation
of the integral $\int f(x)h(X-x)\,dx$: without it the answer scales with how finely you
sampled, which is a common and silent error.

**The seven sample points.** `positions1` spans from $-d$, where the pulses first touch,
to $+d$, where they last touch. Outside that range the overlap is zero, so those seven
shifts cover the entire interesting region.

In [ ]:
# ==========================================
# 1. Define the Two Input Shapes
# ==========================================
width1 = 2.0   # the pulse width d; both pulses are the same width


def get_f(t):
    """Input shape f(t). A rect of width d, centered on the origin."""
    return np.where(np.abs(t) <= width1 / 2, 1.0, 0.0)


def get_h(t):
    """Input shape h(t). The same rect, so the convolution is a triangle."""
    return np.where(np.abs(t) <= width1 / 2, 1.0, 0.0)


# ==========================================
# 2. Perform The "Actual" Convolution
# ==========================================

# Define resolution and grid
dt = 0.002
t_grid = np.arange(-6, 6, dt)

f_arr = get_f(t_grid)
h_arr = get_h(t_grid)

# Calculate Convolution using NumPy's built-in signal processing function
conv1 = np.convolve(f_arr, h_arr, mode='full') * dt

# Construct the correct axis for the resulting convolution
X_start = t_grid[0] + t_grid[0]
X_conv = np.arange(len(conv1)) * dt + X_start

# ==========================================
# 3. Setup Visualization for the Seven Positions
# ==========================================

# Seven snapshots, evenly spaced from first touch to last contact
positions1 = np.linspace(-width1, width1, 7)
labels1 = [r"$X_%d$" % (i + 1) for i in range(7)]

# The value of the convolution at each of those positions
sampled1 = np.interp(positions1, X_conv, conv1)


## Plot

The figure has eight panels. The **top seven** are snapshots: at each shift $X$, the red
curve is the fixed $f(x)$, the blue dashed curve is the shifted $h(X-x)$, and the purple
shading is their product, the integrand. The area of that shaded region *is* the value of
the convolution at that shift, and each panel's title reports it.

The **bottom panel** collects those seven numbers. The black curve is the full convolution
computed at every shift, the red dots mark the seven snapshots above, and the dotted lines
connect each dot to its shift. Reading the seven titles in order gives 0.00, 0.66, 1.33,
2.00, 1.34, 0.67, 0.00: the triangle, one number at a time, before you see it drawn.

Note the reversal in `get_h(X - t_grid)`. Convolution flips the sliding function before
shifting it. Here $h$ is symmetric so the flip is invisible, which is exactly why
convolution and correlation are so easily confused.

In [ ]:
# ==========================================
# 4. Plotting
# ==========================================

# constrained layout keeps the result panel's title clear of the seventh
# panel's x-label, which a fixed hspace does not once height_ratios differ
fig1, axes1 = plt.subplots(8, 1, figsize=(9, 17),
                           gridspec_kw={'height_ratios': [1] * 7 + [2.6]},
                           layout='constrained')

# --- Top seven panels: the overlap at each position ---
for i, (X, lab) in enumerate(zip(positions1, labels1)):
    ax = axes1[i]
    h_shifted = get_h(X - t_grid)          # the flipped and shifted h
    product = get_f(t_grid) * h_shifted    # the integrand

    ax.plot(t_grid, get_f(t_grid), 'r-', label=r'$f(x)$ (Fixed)')
    ax.plot(t_grid, h_shifted, 'b--', label=r'$h(X - x)$ (Shifted)')
    ax.fill_between(t_grid, product, color='purple', alpha=0.3,
                    label='Overlap Product')

    ax.set_title(f'{lab}:   X = {X:+.2f},   area = {sampled1[i]:.2f}',
                 fontsize=13)
    ax.grid(True, linestyle=':', alpha=0.6)
    ax.set_xlim(-4.2, 4.2)
    ax.set_ylim(0, 1.35)
    ax.set_ylabel('amplitude', fontsize=11)
    if i == 0:
        ax.legend(loc='upper right', fontsize=10)
    if i == 6:
        ax.set_xlabel(r'position $(x)$', fontsize=12)

# --- Bottom panel: the convolution result ---
axg = axes1[7]
axg.plot(X_conv, conv1, 'k-', alpha=0.8, label=r'$g(X) = (f \otimes h)(X)$')
axg.plot(positions1, sampled1, 'ro', markersize=8, label='The seven positions')
for X, y in zip(positions1, sampled1):
    axg.plot([X, X], [0, y], 'r:', linewidth=1.5)

axg.set_title('Convolution Output: a triangle of width $2d$')
axg.set_xlabel(r'shift $(X)$')
axg.set_ylabel('amplitude')
axg.legend(loc='upper right', fontsize=12)
axg.grid(True, linestyle=':', alpha=0.6)
axg.set_xlim(-4.2, 4.2)
axg.set_ylim(0, width1 * 1.25)

plt.show()


### Save plot

Commented out on purpose. Uncomment it to regenerate the figure used in the chapter;
leaving it live would rewrite that file every time the notebook is run.

In [ ]:
# fig1.savefig("../figures/ch05/ch05-14-convolution-two-pulses.jpg", dpi=200, bbox_inches='tight')


## Make interactive with ipywidgets

The seven snapshots above are a sampled version of a continuous process. Drag the
slider to move the shifted pulse yourself and watch the overlap area, shown shaded on
top, trace out the triangle below.

Two things worth pausing on as you drag:

- The overlap is a **rectangle** whose width shrinks linearly with $|X|$. That linear
  shrinking is the whole reason the answer is a triangle rather than a curve.
- The area is zero as soon as $|X| > d$, because the pulses no longer touch. That is
  the convolution's **finite support**: convolving two functions of width $d$ gives a
  result of width $2d$, and nothing beyond it.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# --- 1. Global Style Settings ---
plot_params2 = {
    'font.size': 16,
    'font.family': 'sans-serif',
    'axes.linewidth': 1.0,
    'lines.linewidth': 2.5,
    'figure.autolayout': True,
}
plt.rcParams.update(plot_params2)

# --- 2. Define Math Functions ---
width2 = 2.0


def f0(t):
    return np.where(np.abs(t) <= width2 / 2, 1.0, 0.0)


def h0(t):
    return np.where(np.abs(t) <= width2 / 2, 1.0, 0.0)


# --- 3. Pre-calculate Static Data (The Background Curve) ---
tau_grid = np.linspace(-6, 6, 2000)    # High res for integration
X_range = np.linspace(-3, 3, 300)      # Range for the result plot

g_curve = []
for X in X_range:
    # Integration: Area of f(tau) * h(X - tau)
    val = np.trapezoid(f0(tau_grid) * h0(X - tau_grid), tau_grid)
    g_curve.append(val)

# --- 4. Setup Figure and Axes Objects (Object Oriented) ---
fig2, (ax21, ax22) = plt.subplots(2, 1, figsize=(9, 8))
plt.subplots_adjust(hspace=0.4)
fig2.canvas.header_visible = False

# -- Top Plot: The Overlap Process --
ax21.plot(tau_grid, f0(tau_grid), 'r-', label=r'$f(x)$ (Fixed)')
line_h, = ax21.plot([], [], 'b--', label=r'$h(X - x)$ (Shifted)')
fill_collection2 = None

ax21.set_title('Overlap at Shift X')
ax21.set_xlabel(r'position $(x)$')
ax21.set_ylabel('amplitude')
ax21.legend(loc='upper right', fontsize=12)
ax21.grid(True, linestyle=':', alpha=0.6)
ax21.set_xlim(-4.2, 4.2)
ax21.set_ylim(0, 1.35)

# -- Bottom Plot: The Result g(X) --
ax22.plot(X_range, g_curve, 'k-', alpha=0.6, label=r'$g(X)$ (Total)')
line_dot2, = ax22.plot([], [], 'ro', markersize=8, label='Current Value')
line_vert2, = ax22.plot([], [], 'r:', linewidth=1.5)

ax22.set_title('Convolution Output g(X)')
ax22.set_xlabel(r'shift $(X)$')
ax22.set_ylabel('amplitude')
ax22.legend(loc='upper right', fontsize=12)
ax22.grid(True, linestyle=':', alpha=0.6)
ax22.set_xlim(-4.2, 4.2)
ax22.set_ylim(0, width2 * 1.25)

# Prevent the static figure from displaying twice
plt.close(fig2)


# --- 5. The Optimized Update Function ---
def update_plot2(X):
    global fill_collection2

    # A. Calculate Dynamic Data
    h_shifted = h0(X - tau_grid)
    product = f0(tau_grid) * h_shifted
    g_val = np.trapezoid(product, tau_grid)

    # B. Update Artists (Top Plot)
    line_h.set_data(tau_grid, h_shifted)
    if fill_collection2:
        fill_collection2.remove()
    fill_collection2 = ax21.fill_between(tau_grid, product,
                                         color='purple', alpha=0.3)
    ax21.set_title(f'Overlap at Shift X = {X:.2f},   area = {g_val:.3f}')

    # C. Update Artists (Bottom Plot)
    line_dot2.set_data([X], [g_val])
    line_vert2.set_data([X, X], [0, g_val])

    display(fig2)


# --- 6. The Slider ---
widgets.interact(update_plot2,
                 X=widgets.FloatSlider(value=0.0, min=-3.0, max=3.0, step=0.05,
                                       description='Shift X',
                                       continuous_update=False));
